* Generate the dataset and convert it into dataframe

In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv("health_dataset.csv")

In [4]:
# fetch the sample from the population
np.random.seed(0)
sample_data=df.sample(n=200)

1. Formulate the two hypotheses from the dataset

* H₀: There is no significant difference between pre-test and post-test scores (mean difference = 0).
* H₁: There is a significant improvement (mean post-test scores > mean pre-test scores).

2. Calculate Confidence Intervals for key numerical data like age, weight, etc.
3. Find the Critical Value and p-value to interpret the results.

In [65]:
# Hypothesis for age 
# h0: - The population mean age is equal to 50
# h1: - The population mean age is not equal to 50

# find sample age mean
age_sample_mean=round(sum(sample_data["age"])/len(sample_data["age"]),3)

# find sample age standard deviation
age_sample_var=0
var_upper_part=0

for i in sample_data["age"]:
    var_upper_part+=(i-age_sample_mean)**2

var_lower_part=len(sample_data["age"])-1
age_sample_var=var_upper_part/var_lower_part
age_sample_std=np.sqrt(age_sample_var)

# find square root of n
squre_root_n=np.sqrt(len(sample_data["age"]))

# find t value
dof=len(sample_data["age"])-1
critical_value=2.8385

age_population_mean=50

t_value=round((age_sample_mean-age_population_mean)/(age_sample_std/np.sqrt(len(sample_data["age"]))),3)

if abs(t_value) > critical_value:
    print("We reject H₀. There is significant evidence that the population mean age is not equal to 50.")
else:
    print("We fail to reject H₀. There is not enough evidence to suggest that the population mean age is different from 50.")

# find confidence interval
CI_lower_bound=round(age_sample_mean-(critical_value*(age_sample_std/squre_root_n)),2)
CI_upper_bound=round(age_sample_mean+(critical_value*(age_sample_std/squre_root_n)),2)
CI_range=[CI_lower_bound, CI_upper_bound]
print(f"\nConfidence Interval range of Age is {CI_range}")

We fail to reject H₀. There is not enough evidence to suggest that the population mean age is different from 50.

Confidence Interval range of Age is [48.71, 57.07]


4. Perform z or t test based on sample size (mean comparison across groups).

In [ ]:
# Due to i don't have population standard deviation i will perform two sample t test because i have two samples
# i will take gender sample and blood pressure samples and it's hypothesis is given below
# h0: - male blood pressure is greater than or equal to female blood pressure
# h1: - male blood pressure is lower than female blood pressure

# given two samples
gender_sample=sample_data["gender"]
blood_pressure_sample=sample_data["blood_pressure"]

# get the blood_pressure for each gender
male_bp=sample_data[sample_data["gender"]=="Male"]["blood_pressure"]
female_bp=sample_data[sample_data["gender"]=="Female"]["blood_pressure"]

# find the mean of blood_pressure gender-wise
male_bp_mean=round(sample_data[sample_data["gender"]=="Male"]["blood_pressure"].mean(),2)
female_bp_mean=round(sample_data[sample_data["gender"]=="Female"]["blood_pressure"].mean(),2)

# find the std of blood_pressure gender-wise 
male_bp_var=sample_data[sample_data["gender"]=="Male"]["blood_pressure"].var()
female_bp_var=sample_data[sample_data["gender"]=="Female"]["blood_pressure"].var()

# perform the t test
t_value=(male_bp_mean-female_bp_mean)/(np.sqrt((male_bp_var/len(male_bp))+(female_bp_var/len(female_bp))))

dof = (((male_bp_var / len(male_bp)) + (female_bp_var / len(female_bp))) ** 2 / (((male_bp_var / len(male_bp)) ** 2) / (len(male_bp) - 1) + ((female_bp_var / len(female_bp)) ** 2) / (len(female_bp) - 1)))

critical_value=1.6525

if t_value < critical_value:
    print("We fail to reject H₀. There is no significant evidence to suggest that male blood pressure is lower than female blood pressure.")
else:
    print("We reject H₀. There is significant evidence to suggest that male blood pressure is lower than female blood pressure.")

We fail to reject H₀. There is no significant evidence to suggest that male blood pressure is lower than female blood pressure.


5. Conduct a chi-square test on categorical data (e.g., smoking habit Vs disease)

In [34]:
# I will perform chi-square test on smoking habit Vs hypertension data
# Here is the hypothesis: - 
# h0: - Smoking status and hypertension are independent
# h1: - Smoking habit and hypertension are associated

# find the observed frequencies
observed_fequencies=pd.crosstab(sample_data["smoking_status"],sample_data["hypertension"])

row_totals=observed_fequencies.sum(axis=1)
col_totals=observed_fequencies.sum(axis=0)
grand_total=observed_fequencies.values.sum()

# calculate the expected frequencies
expected_frequencies = pd.DataFrame(
    [[(row_total * col_total) / grand_total for col_total in col_totals] for row_total in row_totals],
    index=observed_fequencies.index,
    columns=observed_fequencies.columns
)

# calculate the chi-square value
chi_square_value=((observed_fequencies-expected_frequencies)**2/(expected_frequencies)).sum().sum()

# calculate the dof
total_rows=observed_fequencies.shape[0]
total_cols=observed_fequencies.shape[1]
dof=(total_rows-1)*(total_cols-1)

# find the critical value
critical_value=5.991

# compare the chi-square value and critical value to draw conclusion
if chi_square_value<critical_value:
    print("We fail to reject H₀. There is no significant association between smoking status and hypertension. The two variables appear to be independent.")
else:
    print("We reject H₀. There is a significant association between smoking status and hypertension. The two variables are not independent.")

We fail to reject H₀. There is no significant association between smoking status and hypertension. The two variables appear to be independent.


6. Perform an ANOVA test to check if age group significantly differ in diabetes

In [ ]:
# I will perform ANOVA test on age group Vs cholesterol level data
# Here is the hypothesis: - 
# h0: -  Mean cholesterol levels are equal across all age groups.
# h1: - At least one age group has a different mean cholesterol level.

# group data by age group
groups=sample_data.groupby("age_group")["cholesterol_level"]

# calculate overall mean
overall_mean=sample_data["cholesterol_level"].mean()

# Calculate SSB
SSB=0
for group_name, values in groups:
    n=len(values)
    group_mean=values.mean()
    SSB+=n*((group_mean-overall_mean)**2)

# Calculate SSW
SSW=0
for group_name, values in groups:
    group_mean=values.mean()
    SSW+=((values-group_mean)**2).sum()

# Calculate degree of freedom
k=sample_data["age_group"].nunique()
n=len(sample_data)

df_between=k-1
df_within=n-k

# Calculate mean squares
MSB=SSB/df_between
MSW=SSW/df_within

# Calculate F statistics value
f_value=MSB/MSW

# find the critical value
critical_value=2.37

# compare f_value and critical value to draw conclusion
if f_value<critical_value:
    print("We fail to reject H₀, There is no significant difference in average cholesterol levels are equal across all age groups.")
else:
    print("We reject H₀, There is significant difference in average cholesterol levels are not equal across all age groups.")

We fail to reject H₀, There is no significant difference average cholesterol levels are equal across all age groups.


7. Calculate Covariance and Correlation between continuous variables (e.g., Age Vs BMI)

In [9]:
# calculate mean
age_mean=sample_data["age"].mean()
bmi_mean=sample_data["bmi"].mean()

# calculate numinator for covariance
diff_age_data_and_mean=[]
diff_bmi_data_and_mean=[]
for i in sample_data["age"]:
    diff_age_data_and_mean.append(i-age_mean)
for i in sample_data["bmi"]:
    diff_bmi_data_and_mean.append(i-bmi_mean)

numinator=0
for age,bmi in zip(sample_data["age"],sample_data["bmi"]):
    numinator += (age-age_mean)*(bmi-bmi_mean)

# calculate denominator for covariance
denominator=len(sample_data)

# calculate covariance
covariance=numinator/denominator

# calculate standard deviation of age and bmi
age_std=sample_data["age"].std()
bmi_std=sample_data["bmi"].std()

# calculate correlation
correlation=covariance/(age_std*bmi_std)

if correlation>0:
    print("There's positive linear relationship between age and bmi!")
elif correlation==0:
    print("There's no relationship between age and bmi!")
else:
    print("There's negative linear relationship between age and bmi!")

There's negative linear relationship between age and bmi!
